## Export Gold + clustering output as single-file Parquet
coalesce(1) forces each table to write as one file instead of Spark's usual many-part layout, since these are headed out of Databricks entirely rather than staying in the Spark ecosystem.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "transfermarkt"

tables_to_export = [
    "dim_player", "dim_club", "dim_competition", "dim_date",
    "fact_game", "fact_appearance", "fact_player_valuation", "fact_transfer",
]

for t in tables_to_export:
    df = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/{t}")
    df.coalesce(1).write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/export/{t}")
    print(f"{t}: {df.count()} rows")

clusters = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/player_style_clusters")
clusters.coalesce(1).write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/export/player_style_clusters")
print("player_style_clusters:", clusters.count(), "rows")

## Setup
Credentials come from the clickhouse secret scope, never hardcoded — this notebook is tracked in Git, and secrets in notebook source would end up in your GitHub history.

In [0]:
%pip install clickhouse-connect --quiet

In [0]:
%restart_python

In [0]:
import clickhouse_connect

CATALOG = "workspace"
SCHEMA = "transfermarkt"

client = clickhouse_connect.get_client(
    host=dbutils.secrets.get(scope="clickhouse", key="host"),
    port=8443,
    username=dbutils.secrets.get(scope="clickhouse", key="user"),
    password=dbutils.secrets.get(scope="clickhouse", key="password"),
    secure=True,
)
print(client.command("SELECT 1"))

## Database and tables
DROP + CREATE rather than CREATE IF NOT EXISTS — this notebook will get rerun as the project evolves, and this makes every run start from a clean, correctly-shaped table rather than risking a schema mismatch or duplicate rows from a half-finished previous attempt. Same column definitions and ORDER BY choices as designed earlier.

In [0]:
client.command("CREATE DATABASE IF NOT EXISTS transfermarkt")

ddl = {
"dim_player": """
    CREATE TABLE transfermarkt.dim_player (
        player_key Int32, player_id Nullable(Int32), name Nullable(String),
        first_name Nullable(String), last_name Nullable(String), date_of_birth Nullable(Date32),
        country_of_birth Nullable(String), country_of_citizenship Nullable(String),
        position Nullable(String), sub_position Nullable(String), foot Nullable(String),
        height_in_cm Nullable(Int32), current_club_id Nullable(Int32), current_club_name Nullable(String),
        international_caps Nullable(Int32), international_goals Nullable(Int32),
        market_value_in_eur Nullable(Float64), highest_market_value_in_eur Nullable(Float64)
    ) ENGINE = MergeTree ORDER BY player_key
""",
"dim_club": """
    CREATE TABLE transfermarkt.dim_club (
        club_key Int32, club_id Nullable(Int32), name Nullable(String), club_code Nullable(String),
        domestic_competition_id Nullable(String), stadium_name Nullable(String), stadium_seats Nullable(Int32),
        coach_name Nullable(String), squad_size Nullable(Int32), average_age Nullable(Float64),
        total_market_value Nullable(String), net_transfer_record Nullable(String)
    ) ENGINE = MergeTree ORDER BY club_key
""",
"dim_competition": """
    CREATE TABLE transfermarkt.dim_competition (
        competition_key Int32, competition_id Nullable(String), name Nullable(String),
        type Nullable(String), sub_type Nullable(String), country_name Nullable(String),
        confederation Nullable(String), domestic_league_code Nullable(String), total_clubs Nullable(Int32)
    ) ENGINE = MergeTree ORDER BY competition_key
""",
"dim_date": """
    CREATE TABLE transfermarkt.dim_date (
        date_key Int32, full_date Nullable(Date), year Nullable(Int32), quarter Nullable(Int32),
        month Nullable(Int32), month_name Nullable(String), day Nullable(Int32), day_of_week Nullable(String)
    ) ENGINE = MergeTree ORDER BY date_key
""",
"fact_game": """
    CREATE TABLE transfermarkt.fact_game (
        game_id Int32, date_key Int32, home_club_key Int32, away_club_key Int32, competition_key Int32,
        season Nullable(Int32), round Nullable(String), home_club_goals Nullable(Int32), away_club_goals Nullable(Int32),
        attendance Nullable(Int32), aggregate Nullable(String), competition_type Nullable(String)
    ) ENGINE = MergeTree ORDER BY (competition_key, date_key)
""",
"fact_appearance": """
    CREATE TABLE transfermarkt.fact_appearance (
        appearance_id String, game_id Int32, date_key Int32, player_key Int32, club_key Int32, competition_key Int32,
        goals Nullable(Int32), assists Nullable(Int32), yellow_cards Nullable(Int32),
        red_cards Nullable(Int32), minutes_played Nullable(Int32)
    ) ENGINE = MergeTree ORDER BY (competition_key, date_key, player_key)
""",
"fact_player_valuation": """
    CREATE TABLE transfermarkt.fact_player_valuation (
        date_key Int32, player_key Int32, club_key Int32, market_value_in_eur Nullable(Float64)
    ) ENGINE = MergeTree ORDER BY (player_key, date_key)
""",
"fact_transfer": """
    CREATE TABLE transfermarkt.fact_transfer (
        date_key Int32, player_key Int32, from_club_key Int32, to_club_key Int32,
        transfer_season Nullable(String), transfer_fee Nullable(Float64),
        market_value_in_eur Nullable(Float64), transfer_type Nullable(String)
    ) ENGINE = MergeTree ORDER BY (date_key, from_club_key, to_club_key)
""",
"player_style_clusters": """
    CREATE TABLE transfermarkt.player_style_clusters (
        player_key Int32, position Nullable(String), sub_position Nullable(String),
        goals_per90 Nullable(Float64), assists_per90 Nullable(Float64), yellow_per90 Nullable(Float64),
        minutes_per_appearance Nullable(Float64), cluster Int32, cluster_label String
    ) ENGINE = MergeTree ORDER BY (cluster, player_key)
""",
}

for table, stmt in ddl.items():
    client.command(f"DROP TABLE IF EXISTS transfermarkt.{table}")
    client.command(stmt)
print("Tables created:", list(ddl.keys()))

## Write every table, Spark to ClickHouse, no intermediate files
gold_tables map to /gold/, ml_tables map to /ml/ — same source paths used throughout the project. fact_appearance is the largest at 1.89M rows; if toPandas() hangs or errors here specifically, that's the one to come back and ask about batching, not a sign anything upstream is wrong.

In [0]:
import time

gold_tables = ["dim_player", "dim_club", "dim_competition", "dim_date",
               "fact_game", "fact_appearance", "fact_player_valuation", "fact_transfer"]
ml_tables = ["player_style_clusters"]

for t in gold_tables:
    start = time.time()
    df = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/{t}").toPandas()
    client.insert_df(f"transfermarkt.{t}", df)
    print(f"{t}: {len(df)} rows written in {time.time() - start:.1f}s")

for t in ml_tables:
    start = time.time()
    df = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/{t}").toPandas()
    client.insert_df(f"transfermarkt.{t}", df)
    print(f"{t}: {len(df)} rows written in {time.time() - start:.1f}s")

## Verify counts, straight from the notebook
No need to switch to the Cloud console for this — client.query runs the same check we'd have run there.

In [0]:
result = client.query("""
    SELECT 'dim_player', count() FROM transfermarkt.dim_player
    UNION ALL SELECT 'dim_club', count() FROM transfermarkt.dim_club
    UNION ALL SELECT 'dim_competition', count() FROM transfermarkt.dim_competition
    UNION ALL SELECT 'dim_date', count() FROM transfermarkt.dim_date
    UNION ALL SELECT 'fact_game', count() FROM transfermarkt.fact_game
    UNION ALL SELECT 'fact_appearance', count() FROM transfermarkt.fact_appearance
    UNION ALL SELECT 'fact_player_valuation', count() FROM transfermarkt.fact_player_valuation
    UNION ALL SELECT 'fact_transfer', count() FROM transfermarkt.fact_transfer
    UNION ALL SELECT 'player_style_clusters', count() FROM transfermarkt.player_style_clusters
""")
for row in result.result_rows:
    print(row)